# Production Screener — Multi-Factor Stock Screener

**Philosophy:** Fraud/distress removal IS the alpha. We don't pick stocks — we remove
the ones that will blow up, then let cheap+quality compound.

**Pipeline:** Hard Gates → ML Scoring → Agreement Filter → Top 15 Equal-Weight

**Targets:** 3y CAGR >30%, Sharpe >1.0, small-cap focus ($50M–$2B)

---

## 1. Data Import

In [ ]:
import sys
from pathlib import Path

import joblib
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from pipeline.feature_library import add_piotroski_ext, add_normalised_ratios

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)

In [ ]:
DATA_PATH = ROOT / 'data' / 'historical_dataset_clean.parquet'
MODELS_DIR = ROOT / 'models'

df_raw = pd.read_parquet(DATA_PATH)
df = df_raw[df_raw['period_type'] == 'annual'].copy()

# Enrich: Piotroski extended signals + normalised ratios
df = add_piotroski_ext(df)
df = add_normalised_ratios(df)

# Use latest complete fiscal year
LATEST_YEAR = df[df['market'] == 'US']['fiscal_year'].value_counts().sort_index()
LATEST_YEAR = LATEST_YEAR[LATEST_YEAR >= 100].index.max()
print(f'Dataset: {len(df):,} rows | Using fiscal year: {LATEST_YEAR}')
print(f'US stocks in {LATEST_YEAR}: {len(df[(df["market"]=="US") & (df["fiscal_year"]==LATEST_YEAR)]):,}')

## 2. Hard Gates — Fraud & Distress Removal

This is where the alpha lives. Remove fraud, distress, and uninvestable names BEFORE scoring.

In [ ]:
# Start with latest-year US stocks
universe = df[(df['market'] == 'US') & (df['fiscal_year'] == LATEST_YEAR)].copy()
print(f'Starting universe: {len(universe)} US stocks')

# Gate 1: Beneish M-score < -1.78 (likely NOT a manipulator)
gate1 = universe[universe['beneish_m_score'] < -1.78]
print(f'After Beneish < -1.78:  {len(gate1)} ({len(universe)-len(gate1)} removed — fraud risk)')

# Gate 2: Piotroski F-score >= 3 (minimum financial health)
gate2 = gate1[gate1['piotroski_f_score'] >= 3]
print(f'After Piotroski >= 3:   {len(gate2)} ({len(gate1)-len(gate2)} removed — weak fundamentals)')

# Gate 3: ROA positive (profitable operations)
gate3 = gate2[gate2['piotroski_roa_pos'] == 1]
print(f'After ROA positive:     {len(gate3)} ({len(gate2)-len(gate3)} removed — unprofitable)')

# Gate 4: Not a known fraud suspect
gate4 = gate3[gate3['fraud_suspect'] == 0]
print(f'After fraud_suspect=0:  {len(gate4)} ({len(gate3)-len(gate4)} removed — suspected fraud)')

# Gate 5: Market cap >= $50M (investable)
gate5 = gate4[gate4['market_cap_at_filing'] >= 50_000_000]
print(f'After mkt_cap >= $50M:  {len(gate5)} ({len(gate4)-len(gate5)} removed — micro-cap)')

# Gate 6: Altman Z-score > 1.0 (not in extreme distress)
gate6 = gate5[gate5['altman_z_score'] > 1.0]
print(f'After Altman Z > 1.0:   {len(gate6)} ({len(gate5)-len(gate6)} removed — distress zone)')

survivors = gate6.copy()
print(f'\n✓ {len(survivors)} survivors pass all hard gates ({len(universe)-len(survivors)} eliminated)')

## 3. Feature Selection — 22 Canonical Features (3y Horizon)

In [ ]:
# Load canonical feature set from model metadata
with open(MODELS_DIR / 'model_meta.json') as f:
    meta = json.load(f)

FEATURES_3Y = meta['3y']['features']
print(f'3y LightGBM features ({len(FEATURES_3Y)}):')
for i, feat in enumerate(FEATURES_3Y, 1):
    print(f'  {i:2d}. {feat}')

In [ ]:
# Check feature availability in survivors
missing = [f for f in FEATURES_3Y if f not in survivors.columns]
if missing:
    print(f'WARNING: Missing features: {missing}')
else:
    print(f'✓ All {len(FEATURES_3Y)} features available')

# Median-impute missing values (same as training)
train_medians = meta['3y']['train_medians']
for feat in FEATURES_3Y:
    if feat in survivors.columns:
        median_val = train_medians.get(feat, 0)
        survivors[feat] = survivors[feat].fillna(median_val)

null_pct = survivors[FEATURES_3Y].isnull().sum().sum() / (len(survivors) * len(FEATURES_3Y))
print(f'Post-imputation null rate: {null_pct:.1%}')

## 4. Model Scoring — LightGBM + Decision Tree

In [ ]:
# Load production models
lgbm_3y = joblib.load(MODELS_DIR / 'model_3y.joblib')
tree_dict = joblib.load(MODELS_DIR / 'decision_tree_model.joblib')
tree_model = tree_dict['tree']
tree_features = tree_dict['features']
TREE_THRESHOLD = 0.45  # Production gate (session 42)

print(f'LightGBM 3y: {lgbm_3y.n_estimators_} trees, {len(FEATURES_3Y)} features')
print(f'Decision tree: depth {tree_model.get_depth()}, {len(tree_features)} features')
print(f'Agreement threshold: {TREE_THRESHOLD}')

In [ ]:
# Score with LightGBM (P(beat market over 3y))
X_lgbm = survivors[FEATURES_3Y].values
survivors['ml_3y'] = lgbm_3y.predict_proba(X_lgbm)[:, 1]

# Score with decision tree (agreement signal)
# Impute tree features with 0 for any missing
tree_missing = [f for f in tree_features if f not in survivors.columns]
for f in tree_missing:
    survivors[f] = 0
X_tree = survivors[tree_features].fillna(0).values
survivors['tree_prob'] = tree_model.predict_proba(X_tree)[:, 1]

print(f'Scored {len(survivors)} stocks')
print(f'ml_3y range: [{survivors["ml_3y"].min():.3f}, {survivors["ml_3y"].max():.3f}]')
print(f'tree_prob range: [{survivors["tree_prob"].min():.3f}, {survivors["tree_prob"].max():.3f}]')

## 5. Agreement Gate — Tree Probability >= 0.45

In [ ]:
agreed = survivors[survivors['tree_prob'] >= TREE_THRESHOLD].copy()
print(f'Agreement gate (tree_prob >= {TREE_THRESHOLD}): {len(agreed)} stocks pass')
print(f'  Removed: {len(survivors) - len(agreed)} stocks where tree disagrees')

## 6. Portfolio Construction — Top 15 Equal-Weight

In [ ]:
TOP_N = 15
AUM = 200_000  # $200K retail AUM
LIQUIDITY_PCT = 0.01  # Max 1% of daily volume

# Rank by ml_3y probability (higher = more likely to beat market)
portfolio = agreed.nlargest(TOP_N, 'ml_3y').copy()

# Equal-weight allocation
portfolio['weight'] = 1.0 / min(TOP_N, len(portfolio))
portfolio['position_size'] = AUM * portfolio['weight']

# Liquidity check: can we deploy 1% of daily volume?
# Using market_cap as proxy (no ADTV column available)
# Rule of thumb: daily turnover ~0.5-1% of market cap for small-caps
portfolio['est_daily_volume_usd'] = portfolio['market_cap_at_filing'] * 0.005
portfolio['liquidity_ok'] = portfolio['position_size'] <= (portfolio['est_daily_volume_usd'] * LIQUIDITY_PCT * 100)

print(f'Portfolio: Top {len(portfolio)} by ml_3y score')
print(f'Equal weight: {portfolio["weight"].iloc[0]:.1%} per position (${portfolio["position_size"].iloc[0]:,.0f})')
liq_fail = (~portfolio['liquidity_ok']).sum()
if liq_fail:
    print(f'⚠ {liq_fail} positions may have liquidity constraints')
else:
    print(f'✓ All positions pass liquidity check for ${AUM:,} AUM')

## 7. Stock Analysis — Key Metrics Per Pick

In [ ]:
DISPLAY_COLS = [
    'ticker', 'name', 'sector', 'ml_3y', 'tree_prob',
    'piotroski_f_score', 'altman_z_score', 'beneish_m_score',
    'market_cap_at_filing', 'book_to_market', 'ocf_to_assets',
    'financing_cashflow_to_assets', 'ps_ratio_sector_pct',
]

analysis = portfolio[DISPLAY_COLS].copy()
analysis['market_cap_M'] = analysis['market_cap_at_filing'] / 1e6
analysis = analysis.drop(columns=['market_cap_at_filing'])

print('='*80)
print('PORTFOLIO PICKS — Detailed Analysis')
print('='*80)
for i, row in analysis.iterrows():
    print(f"\n{'─'*60}")
    print(f"  {row['ticker']} — {row['name']}")
    print(f"  Sector: {row['sector']} | Market Cap: ${row['market_cap_M']:.0f}M")
    print(f"  ML Score: {row['ml_3y']:.3f} | Tree: {row['tree_prob']:.3f}")
    print(f"  Piotroski: {row['piotroski_f_score']:.0f} | Altman Z: {row['altman_z_score']:.2f} | Beneish: {row['beneish_m_score']:.2f}")
    print(f"  Book/Market: {row['book_to_market']:.2f} | OCF/Assets: {row['ocf_to_assets']:.3f}")
    print(f"  Thesis: Cheap (P/S pct={row['ps_ratio_sector_pct']:.0%}), profitable (OCF+), not fraud")

## 8. LLM Summary — Buy Rationale Per Stock

Template-based rationale (no API call required). Each pick gets a structured thesis.

In [ ]:
def generate_rationale(row):
    """Generate buy rationale from quantitative signals."""
    signals = []

    # Valuation
    if row.get('book_to_market', 0) > 0.5:
        signals.append('deep value (B/M > 0.5)')
    elif row.get('book_to_market', 0) > 0.3:
        signals.append('moderate value')
    if row.get('ps_ratio_sector_pct', 1) < 0.3:
        signals.append(f'cheap vs sector (P/S bottom {row["ps_ratio_sector_pct"]:.0%})')

    # Quality
    pio = row.get('piotroski_f_score', 0)
    if pio >= 7:
        signals.append(f'excellent quality (Piotroski {pio:.0f}/9)')
    elif pio >= 5:
        signals.append(f'solid quality (Piotroski {pio:.0f}/9)')

    # Cash flow
    ocf = row.get('ocf_to_assets', 0)
    if ocf > 0.10:
        signals.append(f'strong cash generation ({ocf:.1%} of assets)')
    elif ocf > 0.05:
        signals.append('positive operating cash flow')

    # Safety
    az = row.get('altman_z_score', 0)
    if az > 3.0:
        signals.append(f'very safe (Altman Z={az:.1f})')
    elif az > 2.0:
        signals.append(f'financially stable (Altman Z={az:.1f})')

    # Size
    mcap = row.get('market_cap_at_filing', 0) / 1e6
    if mcap < 500:
        signals.append(f'small-cap ${mcap:.0f}M — under institutional radar')
    elif mcap < 2000:
        signals.append(f'mid-small ${mcap:.0f}M')

    # ML confidence
    ml = row.get('ml_3y', 0)
    if ml > 0.6:
        signals.append(f'high ML confidence ({ml:.0%} beat prob)')
    elif ml > 0.5:
        signals.append(f'ML favourable ({ml:.0%} beat prob)')

    return '; '.join(signals) if signals else 'Passes all quantitative gates'


print('='*80)
print('BUY RATIONALE — Per-Stock Thesis')
print('='*80)
for _, row in portfolio.iterrows():
    rationale = generate_rationale(row)
    print(f"\n{row['ticker']} ({row['name']})")
    print(f"  → {rationale}")

## 9. Final Output — Portfolio Summary Table

In [ ]:
# Build final output table
output = portfolio[['ticker', 'name', 'sector', 'ml_3y', 'tree_prob',
                    'piotroski_f_score', 'altman_z_score', 'beneish_m_score',
                    'market_cap_at_filing']].copy()

output = output.rename(columns={
    'piotroski_f_score': 'piotroski',
    'altman_z_score': 'altman_z',
    'beneish_m_score': 'beneish_m',
    'market_cap_at_filing': 'market_cap',
})

# Expected return: use training positive rate as base, scale by ML probability
# Historical 3y CAGR for top quantile: ~30-35%
BASE_3Y_CAGR = 0.30
output['expected_3y_cagr'] = output['ml_3y'] * BASE_3Y_CAGR / meta['3y']['pos_rate']

output['market_cap'] = output['market_cap'] / 1e6  # Convert to $M

# Sort by ml_3y descending
output = output.sort_values('ml_3y', ascending=False).reset_index(drop=True)
output.index = output.index + 1  # 1-indexed rank

print('='*80)
print(f'PRODUCTION PORTFOLIO — Top {len(output)} Picks (FY{LATEST_YEAR})')
print(f'Strategy: Fraud/distress removal → ML ranking → Tree agreement')
print(f'Config: Beneish<-1.78, Piotroski>=3, ROA+, AltmanZ>1.0, tree>={TREE_THRESHOLD}')
print('='*80)
print()

# Format for display
fmt = output.copy()
fmt['market_cap'] = fmt['market_cap'].apply(lambda x: f'${x:,.0f}M')
fmt['expected_3y_cagr'] = fmt['expected_3y_cagr'].apply(lambda x: f'{x:.1%}')
print(fmt.to_string())

In [ ]:
# Portfolio-level statistics
print('\n' + '='*80)
print('PORTFOLIO STATISTICS')
print('='*80)
print(f'  Positions:          {len(output)}')
print(f'  Avg ML score:       {output["ml_3y"].mean():.3f}')
print(f'  Avg tree prob:      {output["tree_prob"].mean():.3f}')
print(f'  Avg Piotroski:      {output["piotroski"].mean():.1f}')
print(f'  Avg Altman Z:       {output["altman_z"].mean():.2f}')
print(f'  Median market cap:  ${output["market_cap"].median():,.0f}M')
print(f'  Avg expected CAGR:  {output["expected_3y_cagr"].mean():.1%}')
print(f'\n  Sectors:')
for sector, count in output['sector'].value_counts().items():
    print(f'    {sector}: {count}')

print(f'\n  Investment philosophy:')
print(f'    • Fraud/distress removal is the PRIMARY alpha')
print(f'    • Small-cap focus — institutions not fishing here')
print(f'    • Equal-weight, annual rebalance')
print(f'    • OOS backtest: CAGR +25.9%, Sharpe 1.08 (2021-2024)')